# Lab 03: LLM Evaluation Harness from Scratch

**Course 13 — LLM Evaluation & Quality**

This lab demonstrates building a production-grade LLM-as-a-Judge evaluation harness for multi-metric scoring (faithfulness, relevance) and CI quality gates.

In [ ]:
import json
import sys
from dataclasses import dataclass
from typing import List, Dict, Any, Optional

sys.path.append('..')
from labs.common.gateway import LLMGateway, OpenAIProvider

In [ ]:
@dataclass
class TestCase:
    id: str
    prompt: str
    context: str

class LLMJudgeHarness:
    def __init__(self, gateway: Optional[LLMGateway] = None):
        self.gateway = gateway or LLMGateway([OpenAIProvider()])

    def evaluate(self, test_case: TestCase, generated_output: str) -> Dict[str, Any]:
        system_prompt = (
            "You are an impartial AI Quality Judge. Score the output on a scale of 0.0 to 1.0 for:\n"
            "- faithfulness: Is it fully grounded in context?\n"
            "- relevance: Does it address the prompt?\n"
            "Return JSON: {\"faithfulness\": float, \"relevance\": float, \"explanation\": string}"
        )
        user_msg = (
            f"Prompt: {test_case.prompt}\n"
            f"Context: {test_case.context}\n"
            f"Output: {generated_output}"
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ]

        resp = self.gateway.generate(messages=messages, temperature=0.0)
        try:
            return json.loads(resp.content)
        except Exception:
            return {"faithfulness": 0.0, "relevance": 0.0, "explanation": "JSON Parse Error"}

In [ ]:
print("--- Lab 03: Running LLM Eval Harness ---")
harness = LLMJudgeHarness()
tc = TestCase(
    id="test-1",
    prompt="Explain RAG",
    context="RAG stands for Retrieval-Augmented Generation. It combines vector retrieval with LLMs."
)
sample_answer = "RAG is Retrieval-Augmented Generation, combining vector search with language models."

result = harness.evaluate(tc, sample_answer)
print(f"Test Case: {tc.id}")
print(f"Judge Output:\n{json.dumps(result, indent=2)}")